# 第3讲 课程代码案例：Spec-driven 协作建模 × Python 工程化

> 课程：Python 进阶 | 教师：孙青 / 欧阳元新 | 平台：CloudStudio + CodeBuddy
>
> **AI 角色定位 = 受控开发者**：Spec 定义约束 → AI 在边界内实现 → 人审查验收
>
> **贯穿案例**：科研实验数据管理与分析工具（sci_analyzer）

## Part 1：Python 工程化基础

**受众**：已完成前两讲实验、能写数据分析脚本的学生

**前置条件**：有 Python 函数和文件操作基础

**学习目标**：
1. 掌握 Python 包结构（模块化拆分、`__init__.py`、导入机制）
2. 掌握类型注解（typing 模块、函数签名）
3. 掌握异常处理（try/except 层级、自定义异常）
4. 掌握 pytest 单元测试（断言/fixture/参数化）

### 1.1 从单文件到模块化：为什么要拆分

In [11]:
# 演示：一个"能跑但不可维护"的脚本
# 改造前：300 行混在一个文件里
# - 数据加载、清洗、统计、可视化全部混合
# - 想改某一步？先花 10 分钟找到它在哪里
# - 想复用清洗逻辑？只能复制粘贴

# 改造后的结构
good_structure = (
    "sci_analyzer/\n"
    "  __init__.py          # 包入口，导出公开接口\n"
    "  loader.py            # 数据加载（职责单一）\n"
    "  cleaner.py           # 数据清洗（职责单一）\n"
    "  stats_engine.py      # 统计分析（职责单一）\n"
    "  visualizer.py        # 可视化（职责单一）\n"
    "  exceptions.py        # 自定义异常\n"
    "tests/\n"
    "  test_loader.py\n"
    "  test_cleaner.py\n"
    "  conftest.py          # 共享 fixture\n"
    "pyproject.toml           # 项目配置"
)
print(good_structure)
print()
print("原则：一个模块只做一件事（Single Responsibility）")

sci_analyzer/
  __init__.py          # 包入口，导出公开接口
  loader.py            # 数据加载（职责单一）
  cleaner.py           # 数据清洗（职责单一）
  stats_engine.py      # 统计分析（职责单一）
  visualizer.py        # 可视化（职责单一）
  exceptions.py        # 自定义异常
tests/
  test_loader.py
  test_cleaner.py
  conftest.py          # 共享 fixture
pyproject.toml           # 项目配置

原则：一个模块只做一件事（Single Responsibility）


### 1.2 创建包结构：__init__.py

In [12]:
# 创建项目目录
import os
from pathlib import Path

project_dir = Path("sci_analyzer")
project_dir.mkdir(exist_ok=True)

init_content = (
    '"""科研实验数据管理与分析工具"""\n\n'
    'from .loader import load_data\n'
    'from .cleaner import clean_data, remove_duplicates, detect_outliers\n'
    'from .stats_engine import run_statistics\n'
    'from .exceptions import DataError, LoadError, CleanError\n\n'
    '__version__ = "0.1.0"\n'
    '__all__ = [\n'
    '    "load_data", "clean_data", "remove_duplicates",\n'
    '    "detect_outliers", "run_statistics",\n'
    '    "DataError", "LoadError", "CleanError",\n'
    ']\n'
)
(project_dir / "__init__.py").write_text(init_content)

print("__init__.py 作用：")
print("1. 标识这是一个 Python 包")
print("2. 控制外部 import 时能看到什么（__all__）")
print("3. 提供包级别的元数据（__version__）")

__init__.py 作用：
1. 标识这是一个 Python 包
2. 控制外部 import 时能看到什么（__all__）
3. 提供包级别的元数据（__version__）


### 1.3 类型注解：让代码自文档化

In [13]:
from typing import Optional, Any
from pathlib import Path
import pandas as pd

# 无类型注解：函数接口模糊
def process(data, config):
    pass  # 调用者不知道传什么、返回什么

# 有类型注解：一目了然
def load_data(
    file_path: Path,
    sheet_name: Optional[str] = None,
    encoding: str = "utf-8"
) -> pd.DataFrame:
    """加载数据文件。

    Args:
        file_path: 数据文件路径，支持 .csv/.xlsx/.json
        sheet_name: Excel 工作表名（仅 Excel 格式需要）
        encoding: 文件编码（仅 CSV 格式需要）

    Returns:
        加载后的 DataFrame

    Raises:
        FileNotFoundError: 文件不存在
        ValueError: 不支持的文件格式
    """
    pass

# 常用类型注解
from typing import Union

def detect_outliers(
    series: pd.Series,
    method: str = "iqr",
    threshold: float = 1.5
) -> tuple[pd.Series, dict[str, Any]]:
    """返回 (布尔掩码, 统计信息字典)"""
    pass

print("类型注解三大好处：")
print("1. IDE 自动补全更精准")
print("2. AI 生成的代码更容易审查")
print("3. 代码即文档，减少沟通成本")

类型注解三大好处：
1. IDE 自动补全更精准
2. AI 生成的代码更容易审查
3. 代码即文档，减少沟通成本


### 1.4 异常处理：优雅应对错误

In [14]:
# 自定义异常层级
class DataError(Exception):
    """数据处理相关错误的基类"""
    pass

class LoadError(DataError):
    """数据加载失败"""
    pass

class CleanError(DataError):
    """数据清洗失败"""
    def __init__(self, column: str, reason: str):
        self.column = column
        super().__init__(f"列\'{column}\'清洗失败: {reason}")

# 异常处理实战：数据加载
def load_data_demo(file_path: Path) -> pd.DataFrame:
    if not file_path.exists():
        raise FileNotFoundError(f"文件不存在: {file_path}")

    suffix = file_path.suffix.lower()
    if suffix not in ('.csv', '.xlsx', '.xls', '.json'):
        raise ValueError(f"不支持的格式: {suffix}")

    try:
        if suffix == '.csv':
            return pd.read_csv(file_path, encoding='utf-8-sig')
        elif suffix in ('.xlsx', '.xls'):
            return pd.read_excel(file_path)
        else:
            return pd.read_json(file_path)
    except UnicodeDecodeError:
        return pd.read_csv(file_path, encoding='gbk')
    except Exception as e:
        raise LoadError(f"加载 {file_path.name} 失败") from e

# 测试错误处理
try:
    load_data_demo(Path("不存在的文件.csv"))
except FileNotFoundError as e:
    print(f"正确捕获: {e}")

from tempfile import NamedTemporaryFile

with NamedTemporaryFile(suffix=".txt") as unsupported_file:
    try:
        load_data_demo(Path(unsupported_file.name))
    except ValueError as e:
        print(f"正确捕获: {e}")


正确捕获: 文件不存在: 不存在的文件.csv
正确捕获: 不支持的格式: .txt


### 1.5 pytest 单元测试

In [15]:
# 写入测试文件演示
from pathlib import Path
import textwrap

tests_dir = Path("tests")
tests_dir.mkdir(exist_ok=True)
(tests_dir / "__init__.py").touch()

test_code = textwrap.dedent('''\
import pytest
import pandas as pd
import numpy as np
from pathlib import Path

@pytest.fixture
def sample_df():
    return pd.DataFrame({
        "name": ["A", "B", "C", "A", "D"],
        "value": [1.0, 2.0, np.nan, 1.0, 100.0],
        "category": ["x", "y", "x", "x", "y"]
    })

@pytest.fixture
def csv_file(tmp_path):
    f = tmp_path / "test.csv"
    f.write_text("name,value\\nA,1\\nB,2\\nC,3")
    return f

def test_load_csv(csv_file):
    df = pd.read_csv(csv_file)
    assert len(df) == 3
    assert list(df.columns) == ["name", "value"]

def test_remove_duplicates(sample_df):
    result = sample_df.drop_duplicates()
    assert len(result) == 4

def test_load_nonexistent():
    with pytest.raises(Exception):
        pd.read_csv(Path("ghost_file.csv"))

@pytest.mark.parametrize("threshold,expected_outliers", [
    (1.5, 1),
    (3.0, 0),
])
def test_outlier_detection(sample_df, threshold, expected_outliers):
    values = sample_df["value"].dropna()
    Q1, Q3 = values.quantile(0.25), values.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - threshold * IQR, Q3 + threshold * IQR
    outliers = values[(values < lower) | (values > upper)]
    assert len(outliers) == expected_outliers
''')

(tests_dir / "test_demo.py").write_text(test_code)
print("测试文件已生成: tests/test_demo.py")
print()
print("运行方式: pytest tests/ -v")
print("参数化测试: 一组代码覆盖多种输入组合")

测试文件已生成: tests/test_demo.py

运行方式: pytest tests/ -v
参数化测试: 一组代码覆盖多种输入组合


In [23]:
# 运行 pytest
import subprocess
result = subprocess.run(
    ["python", "-m", "pytest", "tests/test_demo.py", "-v", "--tb=short"],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

============================= test session starts ==============================
platform darwin -- Python 3.12.2, pytest-8.2.1, pluggy-1.5.0 -- /Users/jafekin/miniconda3/bin/python
cachedir: .pytest_cache
rootdir: /Users/jafekin/Documents/Python进阶课课件及案例开发/课程资料v3/第3讲
plugins: anyio-4.11.0
collecting ... collected 5 items

tests/test_demo.py::test_load_csv PASSED                                 [ 20%]
tests/test_demo.py::test_remove_duplicates PASSED                        [ 40%]
tests/test_demo.py::test_load_nonexistent PASSED                         [ 60%]
tests/test_demo.py::test_outlier_detection[1.5-1] PASSED                 [ 80%]
tests/test_demo.py::test_outlier_detection[3.0-0] PASSED                 [100%]

============================== 5 passed in 0.29s ===============================



### 1.6 AI 协作练习：TDD + AI

**TDD（测试驱动开发）+ AI 的工作流**：

```
步骤1：你写测试 → 定义"什么是正确的"
步骤2：让 AI 实现 → "请让这些测试通过"
步骤3：运行 pytest → 自动裁判
步骤4：失败？告知 AI 哪个测试失败 → AI 修改 → 重新验证
```

**Prompt 示例**：
```
我已经写好了 tests/test_cleaner.py（见 @tests/test_cleaner.py）。
请实现 sci_analyzer/cleaner.py，使所有测试通过。
遵守规则：类型注解完整、不修改原输入、用 logging 不用 print。
```

## Part 2：Spec-driven 开发范式

**受众**：已掌握 Part 1 工程化基础的学生  
**前置条件**：已能为一个函数编写类型注解、异常处理和 pytest 测试

### 本节要完成的交付物

围绕 `reporter.py` 模块完成一次可复现的 **Spec → 测试 → AI 实现 → 自动验收 → 人工审查** 循环。完成后，工作区中应出现：

```text
spec_driven_demo/
├── constitution.md      # 项目级硬约束
├── context.md           # 环境与已有资产
├── tasks.md             # 功能、边界和验收标准
├── speckit.md           # 交给 AI 的当前任务入口
├── sci_analyzer/
│   └── reporter.py      # AI 按规约实现的模块
└── tests/
    └── test_reporter.py # 人先写好的验收测试
```

> **AI 的角色是受控开发者，不是需求猜测器。** 你先规定不能做什么、现有条件是什么、做到什么算完成；AI 再在这些边界内编码。


### 2.1 先判断：这次该用 Prompt、Rules，还是 Spec？

| 层级 | 你提供的内容 | 典型场景 | AI 自由度 |
|---|---|---|---|
| L1：两段式 Prompt | 一个明确请求 + 必要上下文 | 独立函数、快速原型 | 高 |
| L2：Rules 约束 | 请求 + 团队风格/目录规则 | 多人协作、统一代码风格 | 中 |
| L3：Spec-driven | 硬约束 + 环境 + 任务 + 验收 | 多模块开发、重构、可评分作业 | 低 |

**快速决策**：

```text
只改一个函数，且失败影响可控？        → Prompt + 必要上下文
需要统一命名、日志、目录等通用风格？    → Prompt + Rules
涉及多个模块、已有资产、测试和验收？    → Spec-driven
```

本案例的 `reporter.py` 虽然只有一个模块，但它要接收 DataFrame、写入文件、保证输入不被修改，并且要通过测试；因此使用 L3，把接口、边界和验收条件都固定下来。


### 2.2 三份文档分别解决什么问题？

| 文档 | 回答的问题 | 不能只写成什么样 |
|---|---|---|
| `constitution.md` | **永远不能违反什么？** | “代码写得规范一些”这类不可检查的口号 |
| `context.md` | **项目现在有什么、运行在哪里？** | 让 AI 猜 Python 版本、依赖和输入数据 |
| `tasks.md` | **本次要做什么，怎样算完成？** | 只有“实现一个报告功能”，没有接口与边界 |

对应 Speckit 五步：

1. **constitution**：写下不可违反的项目规则；
2. **specify**：把业务需求写成可观察行为；
3. **plan**：要求 AI 先给模块划分和风险点，人审查后再编码；
4. **tasks**：拆成可以逐项验收的实现任务；
5. **implement**：AI 实现，pytest 和人工审查共同验收。

> 好规约中的每一条关键要求，都应当能在**代码审查**或**自动测试**中找到证据。


### 2.3 实战目标：为 `reporter.py` 写可验收的规约

我们要实现：

```python
generate_report(df, stats_result, output_path) -> Path
```

它把实验数据概览和统计结果写成 Markdown 报告。下面的代码会在本地生成 4 份规约文档。先阅读生成后的 `speckit.md`，再把它作为上下文提交给 AI。

**本轮刻意写清的边界**：

- `df` 必须是 `pandas.DataFrame`；
- 不能修改调用者传入的 `df`；
- 输出必须是 `.md` 文件，父目录不存在时自动创建；
- 报告必须包含固定的 4 个一级标题；
- 统计结果由调用者传入，不能在模块内部“猜测”或重新计算；
- 失败时抛出具体异常，不能用 `except Exception: pass` 掩盖问题。


In [17]:
from pathlib import Path
import textwrap

spec_root = Path("spec_driven_demo")
package_dir = spec_root / "sci_analyzer"
tests_dir = spec_root / "tests"
package_dir.mkdir(parents=True, exist_ok=True)
tests_dir.mkdir(parents=True, exist_ok=True)
(package_dir / "__init__.py").write_text('"""Spec-driven 报告模块示例。"""\n', encoding="utf-8")
(tests_dir / "__init__.py").touch()

constitution = """# constitution.md

## 不可违反的规则

1. 所有公开函数必须有完整类型注解和 Google style docstring。
2. 函数不得修改调用者传入的 `DataFrame` 或统计结果映射。
3. 文件路径使用 `pathlib.Path`；只接受 `.md` 后缀的输出文件。
4. 不允许 bare `except`；参数错误使用 `TypeError` 或 `ValueError` 明确说明原因。
5. 不使用 `print` 作为运行日志；需要日志时使用 `logging`。
6. 数值展示统一保留 4 位小数。
7. 每个公开函数至少有正常路径、边界路径和异常路径测试。
8. 变量名使用英文 `snake_case`，解释性注释使用中文。
"""

context = """# context.md

## 项目环境

- Python 3.10+
- pandas 2.x
- pytest 8.x
- 运行位置：课程第 3 讲目录（CloudStudio/Linux 或本地环境）

## 已有资产

- 输入数据：清洗完成的 `pandas.DataFrame`，列名可能为中文。
- 上游统计模块：已经计算完成并传入 `stats_result`，本模块不重复计算统计量。
- 输出目标：供实验报告直接引用的 Markdown 文件。
"""

tasks = """# tasks.md

## Task：实现 `sci_analyzer/reporter.py`

### 公开接口

```python
def generate_report(
    df: pd.DataFrame,
    stats_result: Mapping[str, float | int],
    output_path: Path,
) -> Path:
    ...
```

### 功能要求

1. 校验 `df` 是 `pandas.DataFrame`，`stats_result` 是映射类型。
2. 校验输出路径后缀为 `.md`；必要时创建父目录。
3. 写出 UTF-8 Markdown 文件，包含四个一级标题：`# 实验数据分析报告`、`# 数据概览`、`# 列信息`、`# 统计结果`。
4. 数据概览至少包含行数、列数；列信息至少列出列名和数据类型。
5. `stats_result` 中的浮点数保留 4 位小数，整数按整数展示。
6. 返回最终写入的 `Path`，且不修改原始 `df`。

## 验收标准

- `python -m pytest spec_driven_demo/tests/test_reporter.py -q` 全部通过；
- 正常写报告、创建父目录、输入不变、错误后缀、错误输入类型均有测试；
- 人工审查没有 bare `except`、硬编码路径和 `print` 调试语句。
"""

speckit = """# speckit.md — Reporter 模块当前任务

请先阅读并遵守：

- @constitution.md
- @context.md
- @tasks.md

## 本轮交付

实现 `sci_analyzer/reporter.py` 中的 `generate_report`。

## 执行顺序（不要跳步）

1. 先给出不超过 8 行的实现计划：参数校验、内容组装、文件写入、测试对应关系。
2. 确认接口不变后，再实现函数。
3. 运行 `python -m pytest spec_driven_demo/tests/test_reporter.py -q`。
4. 若失败，报告失败的测试名、根因和最小修改方案；不要靠删除测试或放宽断言通过。

## 交付前自检

- 类型注解和 Google style docstring 是否完整？
- 是否只读取 `df`，没有原地修改？
- 是否拒绝非 `.md` 输出路径？
- 是否创建了缺失的父目录？
- 是否没有使用 `print` 和 bare `except`？
"""

for filename, content in {
    "constitution.md": constitution,
    "context.md": context,
    "tasks.md": tasks,
    "speckit.md": speckit,
}.items():
    (spec_root / filename).write_text(textwrap.dedent(content).lstrip(), encoding="utf-8")


### 2.4 把 `speckit.md` 交给 AI 前：先固定验收测试（TDD）

先写测试不是为了“为难 AI”，而是为了让“完成”变成可运行的事实。下面的测试覆盖了：

| 测试 | 约束证据 |
|---|---|
| 正常写出报告 | 固定标题、行列信息、数值格式 |
| 输入不变 | 不允许原地修改 `df` |
| 自动创建目录 | 文件 I/O 行为明确 |
| 错误后缀 | `.md` 是硬边界 |
| 错误数据类型 | 参数校验具体且可诊断 |

将下一段代码运行后，先在 CodeBuddy 中粘贴下面的请求，**要求先返回计划，不要立刻写代码**：

```text
@speckit.md
请实现 spec_driven_demo/sci_analyzer/reporter.py。
先只输出实现计划，并逐项说明它如何满足 tests/test_reporter.py 的验收条件；
等待我确认计划后再编码。
```

计划审查通过后，再允许 AI 修改 `reporter.py`。这样可以在代码生成之前发现“接口理解错了”“想重算统计结果”等方向性问题。


In [18]:
import textwrap

reporter_tests = """import pandas as pd
import pytest

from spec_driven_demo.sci_analyzer.reporter import generate_report


@pytest.fixture
def sample_df() -> pd.DataFrame:
    return pd.DataFrame({"样品": ["A", "B"], "浓度": [1.25, 2.5]})


def test_generate_report_writes_required_sections(tmp_path, sample_df) -> None:
    output_path = generate_report(
        sample_df,
        {"mean_concentration": 1.875, "sample_count": 2},
        tmp_path / "report.md",
    )

    content = output_path.read_text(encoding="utf-8")
    assert output_path.exists()
    assert "# 实验数据分析报告" in content
    assert "# 数据概览" in content
    assert "# 列信息" in content
    assert "# 统计结果" in content
    assert "1.8750" in content


def test_generate_report_does_not_mutate_dataframe(tmp_path, sample_df) -> None:
    original = sample_df.copy(deep=True)

    generate_report(sample_df, {"sample_count": 2}, tmp_path / "report.md")

    pd.testing.assert_frame_equal(sample_df, original)


def test_generate_report_creates_parent_directory(tmp_path, sample_df) -> None:
    output_path = tmp_path / "nested" / "reports" / "report.md"

    returned_path = generate_report(sample_df, {"sample_count": 2}, output_path)

    assert returned_path == output_path
    assert output_path.exists()


def test_generate_report_rejects_non_markdown_path(tmp_path, sample_df) -> None:
    with pytest.raises(ValueError, match=".md"):
        generate_report(sample_df, {"sample_count": 2}, tmp_path / "report.txt")


def test_generate_report_rejects_non_dataframe(tmp_path) -> None:
    with pytest.raises(TypeError, match="DataFrame"):
        generate_report([{"样品": "A"}], {"sample_count": 1}, tmp_path / "report.md")
"""

(tests_dir / "test_reporter.py").write_text(
    textwrap.dedent(reporter_tests), encoding="utf-8"
)


1670

### 2.5 审查 AI 的实现：不要只看“能运行”

下面的实现单元模拟 **计划经人工确认后** 的 AI 交付。运行前可先自己阅读它，并用审查清单逐项标注证据位置：

- [ ] 函数签名、返回类型、docstring 是否与任务一致？
- [ ] `df`、`stats_result` 的校验是否在业务逻辑之前？
- [ ] 代码是否只读取输入，而非执行 `inplace=True`、列赋值或排序？
- [ ] 目录创建、UTF-8 编码和 `.md` 后缀是否都被处理？
- [ ] 浮点数是否按 Constitution 统一格式化？
- [ ] 是否存在 `print`、bare `except`、硬编码绝对路径？

**反例提示**：如果 AI 用 `str(output_path).endswith("md")` 判断后缀、在 `df` 上新增临时列、或者用 `except Exception: return None`，即使部分示例能跑，也应退回修改。它们分别违反了可移植性、输入不可变性和错误可诊断性。


In [19]:
reporter_implementation = """\"\"\"将实验数据和统计结果写成 Markdown 报告。\"\"\"

from collections.abc import Mapping
from pathlib import Path

import pandas as pd


def _format_stat(value: float | int) -> str:
    \"\"\"格式化统计值，浮点数保留 4 位小数。\"\"\"
    if isinstance(value, float):
        return f\"{value:.4f}\"
    return str(value)


def generate_report(
    df: pd.DataFrame,
    stats_result: Mapping[str, float | int],
    output_path: Path,
) -> Path:
    \"\"\"生成实验数据分析 Markdown 报告。

    Args:
        df: 已完成清洗的实验数据，不会被本函数修改。
        stats_result: 上游模块计算完成的统计结果。
        output_path: 目标 Markdown 文件路径，必须以 `.md` 结尾。

    Returns:
        实际写入的报告路径。

    Raises:
        TypeError: `df` 不是 DataFrame 或 `stats_result` 不是映射类型。
        ValueError: 输出文件不是 `.md` 格式。
    \"\"\"
    if not isinstance(df, pd.DataFrame):
        raise TypeError(\"df 必须是 pandas.DataFrame\")
    if not isinstance(stats_result, Mapping):
        raise TypeError(\"stats_result 必须是 Mapping 类型\")

    report_path = Path(output_path)
    if report_path.suffix.lower() != \".md\":
        raise ValueError(\"output_path 必须以 .md 结尾\")

    report_path.parent.mkdir(parents=True, exist_ok=True)
    column_lines = [f\"- `{column}`: {dtype}\" for column, dtype in df.dtypes.items()]
    stat_lines = [
        f\"- `{name}`: {_format_stat(value)}\"
        for name, value in stats_result.items()
    ]
    content = \"\\n\".join(
        [
            \"# 实验数据分析报告\",
            \"\",
            \"# 数据概览\",
            f\"- 行数：{df.shape[0]}\",
            f\"- 列数：{df.shape[1]}\",
            \"\",
            \"# 列信息\",
            *column_lines,
            \"\",
            \"# 统计结果\",
            *(stat_lines or [\"- 未提供统计结果\"]),
            \"\",
        ]
    )
    report_path.write_text(content, encoding=\"utf-8\")
    return report_path
"""

(package_dir / "reporter.py").write_text(
    textwrap.dedent(reporter_implementation), encoding="utf-8"
)


1727

### 2.6 自动验收 + 人工验收：两道关都不能少

运行下方代码会执行规约中的 pytest 命令；测试失败时，Notebook 会保留完整的失败输出，方便把**失败测试名、断言信息和最小修复目标**发回给 AI。不要只说“修一下”。

建议使用下面的反馈模板：

```text
测试未通过：test_generate_report_rejects_non_markdown_path
现象：函数接受了 report.txt。
违反条款：tasks.md 的“输出路径必须为 .md”。
最小修复：在创建目录和写文件之前，使用 Path.suffix 校验后缀并抛出 ValueError。
请只修改 reporter.py，并重新运行指定测试。
```

pytest 全绿后仍需人工审查：测试只能证明已写断言的行为，不能自动判断 AI 是否引入了多余依赖、泄露了敏感数据或把复杂度藏进了不必要的抽象。


In [24]:
import subprocess
import sys

validation = subprocess.run(
    [
        sys.executable,
        "-m",
        "pytest",
        "spec_driven_demo/tests/test_reporter.py",
        "-q",
    ],
    capture_output=True,
    text=True,
)
assert validation.returncode == 0, validation.stdout + "\n" + validation.stderr


### 2.7 本案例复盘：把“约束”变成“可验证证据”

| Spec 条款 | 自动化证据 | 人工审查证据 |
|---|---|---|
| 输出必须是 Markdown | `test_generate_report_rejects_non_markdown_path` | 检查 `Path.suffix`，而不是字符串猜测 |
| 不修改原始数据 | `test_generate_report_does_not_mutate_dataframe` | 检查没有 `inplace=True` 和列赋值 |
| 输出目录可用 | `test_generate_report_creates_parent_directory` | 检查使用 `Path.mkdir(parents=True, exist_ok=True)` |
| 格式稳定 | 标题、`1.8750` 等断言 | 检查格式化规则集中在 `_format_stat` |
| 失败可诊断 | 非 DataFrame 输入的异常断言 | 检查没有 bare `except` 或吞异常 |

**课后迁移**：把本节的 `reporter.py` 换成你自己的 `cleaner.py`、`visualizer.py` 或 CLI 模块时，不要只替换功能描述；还要同步替换输入边界、不可变性要求、异常路径和验收测试。这样 AI 写的是你的工程，而不是一个“看起来像答案”的片段。
